In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

# =========================================================
# 1. Load Data
# =========================================================

train = pd.read_csv('/kaggle/input/competitions/bike-demande-competition/train1222.csv')
test = pd.read_csv('/kaggle/input/competitions/bike-demande-competition/test1222.csv')

# Ensure test file contains an "id" column
if 'id' not in test.columns:
    test.rename(columns={test.columns[0]: 'id'}, inplace=True)

backup_ids = test['id'].copy()

# =========================================================
# 2. Feature Engineering
# =========================================================

train['dt'] = pd.to_datetime(train['started_at'])

# Focus only on December data
dec_train = train[train['dt'].dt.month == 12].copy()

# Identify valid sink stations
sink_stations = set(dec_train['end_station_id'].astype(str).unique())

# Station outgoing trips
st_stats = (
    dec_train.groupby('start_station_id')
    .size()
    .reset_index(name='out_count')
)

# Station incoming trips
in_stats = (
    dec_train.groupby('end_station_id')
    .size()
    .reset_index(name='in_count')
)

# Merge station statistics
st_dna = pd.merge(
    st_stats,
    in_stats,
    left_on='start_station_id',
    right_on='end_station_id',
    how='left'
).fillna(0)

# =========================================================
# 3. Smoothed Return Ratio
# =========================================================

# Stronger Laplace smoothing
st_dna['st_ratio'] = (
    (st_dna['in_count'] + 20) /
    (st_dna['out_count'] + 20)
)

# Restrict extreme ratios
st_dna['st_ratio'] = st_dna['st_ratio'].clip(0.85, 1.45)

# =========================================================
# 4. Station Weight Dampening
# =========================================================

# Raw station activity weight
st_dna['st_weight_raw'] = (
    st_dna['out_count'] /
    st_dna['out_count'].mean()
)

# Reduce extreme station influence
st_dna['st_weight'] = np.where(
    st_dna['st_weight_raw'] > 4,
    4 + np.log1p(st_dna['st_weight_raw'] - 4),
    st_dna['st_weight_raw']
)

st_dna['st_id_str'] = st_dna['start_station_id'].astype(str)

# =========================================================
# 5. Train Linear Regression Model
# =========================================================

daily_counts = (
    dec_train.groupby(dec_train['dt'].dt.date.rename('date'))
    .size()
    .reset_index(name='total')
)

daily_counts['dow'] = pd.to_datetime(
    daily_counts['date']
).dt.dayofweek

# Average trips per station
daily_counts['avg_per_st'] = daily_counts['total'] / 1148

# One-hot encoding for weekdays
X_train = pd.get_dummies(
    daily_counts['dow'],
    prefix='dow'
).astype(float)

y_train = daily_counts['avg_per_st']

# Train model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# =========================================================
# 6. Prediction
# =========================================================

test['dt'] = pd.to_datetime(test['date'])
test['dow'] = test['dt'].dt.dayofweek
test['day'] = test['dt'].dt.day

# Encode weekdays
X_test = pd.get_dummies(
    test['dow'],
    prefix='dow'
).astype(float)

# Align columns with training set
for col in X_train.columns:
    if col not in X_test.columns:
        X_test[col] = 0.0

X_test = X_test[X_train.columns]

# Base demand prediction
test['base_demand'] = lr_model.predict(X_test)

# =========================================================
# 7. Apply Geographic Station Behavior
# =========================================================

test['st_str'] = test['start_station_id'].astype(str)

test = test.merge(
    st_dna[['st_id_str', 'st_weight', 'st_ratio']],
    left_on='st_str',
    right_on='st_id_str',
    how='left'
).fillna(0.5)

# Demand and returns estimation
test['demand'] = (
    test['base_demand'] *
    test['st_weight']
)

test['returns'] = (
    test['demand'] *
    test['st_ratio']
)

# =========================================================
# 8. Safety Filters
# =========================================================

# Penalize unknown sink stations
test.loc[
    ~test['st_str'].isin(sink_stations),
    'returns'
] = 0.01

# End-of-year seasonal adjustments
test.loc[
    test['dow'] == 6,
    ['demand', 'returns']
] *= 0.55

# Christmas Day
test.loc[
    test['day'] == 25,
    ['demand', 'returns']
] *= 0.15

# Post-Christmas recovery
test.loc[
    test['day'] == 26,
    ['demand', 'returns']
] *= 0.45

# Monday recovery effect
test.loc[
    test['day'] == 28,
    'returns'
] *= 1.35

# =========================================================
# 9. Hard Caps
# =========================================================

test['demand'] = test['demand'].clip(upper=32.0)
test['returns'] = test['returns'].clip(upper=35.0)

# =========================================================
# 10. Final Calibration
# =========================================================

test['demand'] *= (
    1.61 / test['demand'].mean()
)

test['returns'] *= (
    1.81 / test['returns'].mean()
)

# =========================================================
# 11. Create Submission File
# =========================================================

submission = pd.DataFrame({
    'id': backup_ids,
    'demand': test['demand'].clip(lower=0.1),
    'returns': test['returns'].clip(lower=0.01)
})

submission.to_csv('submission.csv', index=False)

print(
    f" V210: Demand Mean "
    f"{submission['demand'].mean():.4f} | "
    f"Returns Mean {submission['returns'].mean():.4f}"
)